In [1]:
import os
os.chdir('/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML')
os.getcwd()

'/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML'

In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier


In [6]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import pandas as pd
import matplotlib.pyplot as plt

In [7]:
X = pd.read_csv(
    "data/processed/ml_matrix_top2000.csv.gz",
    index_col=0
)

meta = pd.read_csv(
    "data/processed/pdc_metadata.csv"
)

print(X.shape)
print(meta.shape)

(4846, 2000)
(14430, 58)


In [8]:
meta_rf = meta[
    meta["cell"].isin(X.index)
].copy()

In [9]:
meta_rf = (
    meta_rf
    .set_index("cell")
    .loc[X.index]
    .reset_index()
)

In [10]:
if "index" in meta_rf.columns:
    meta_rf = meta_rf.rename(
        columns={"index": "cell"}
    )

In [11]:
print(X.shape)
print(meta_rf.shape)

meta_rf["is_malignant"].value_counts()

(4846, 2000)
(4846, 58)


is_malignant
Malignant       4405
Premalignant     441
Name: count, dtype: int64

In [12]:
y = meta_rf["is_malignant"].map({
    "Premalignant": 0,
    "Malignant": 1
})

print(y.value_counts())

is_malignant
1    4405
0     441
Name: count, dtype: int64


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(3876, 2000)
(970, 2000)


In [15]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [16]:
xgb.fit(
    X_train,
    y_train
)

,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'
,feature_types,None


In [17]:
pred_xgb = xgb.predict(
    X_test
)

In [18]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred_xgb
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        88
           1       1.00      1.00      1.00       882

    accuracy                           1.00       970
   macro avg       1.00      1.00      1.00       970
weighted avg       1.00      1.00      1.00       970



In [19]:
xgb_importance = pd.DataFrame({
    "gene": X.columns,
    "importance": xgb.feature_importances_
})

xgb_importance = (
    xgb_importance
    .sort_values(
        "importance",
        ascending=False
    )
)

xgb_importance.head(20)

,gene,importance
1555,ALKBH7,0.308069
238,ALOX5AP,0.301360
170,ANKRD12,0.091550
1546,SIGLEC6,0.079173
456,CDC40,0.046734
190,PDLIM1,0.028281
647,SERHL2,0.027785
1902,ERC2,0.022285
983,SCN3A,0.013073
159,HES6,0.010910
